# Chapter 8: Feed-Forward and Norms

[Read this chapter online](https://jackluu.io/book/section-2-attention/ch08-feedforward-and-norms/) &nbsp;|&nbsp; [Open in Colab](https://colab.research.google.com/github/jackluucoding/build-llm-from-zero/blob/main/notebooks/ch08-feedforward-and-norms.ipynb)

From *Building an LLM from Zero: Look Inside the Black Box* by Truong (Jack) Luu.


In [ ]:
# Run me first. Safe to run more than once; it skips whatever is already done.
import os
import subprocess
import sys

FOLDER = "build-llm-from-zero"

# 1. Fetch the code, unless we are already inside it
if os.path.basename(os.getcwd()) != FOLDER:
    if not os.path.isdir(FOLDER):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackluucoding/build-llm-from-zero"], check=True)
    os.chdir(FOLDER)
sys.path.insert(0, os.getcwd())

# 2. PyTorch, the CPU build, which is all this book needs
try:
    import torch
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)

# 3. The Shakespeare text
subprocess.run([sys.executable, "src/utils/download_data.py"], check=True)

print("Ready. Working in", os.getcwd())

# Chapter 8: Feed-Forward and Norms

![Where we are in the pipeline](../assets/diagrams/ch08-where-we-are.png){ width="756" }
*Figure 8.1: We just built attention; now we process that context.*

While attention (from Chapter 7) lets a token gather information from its neighbors, gathering information is only half the job. The token still needs to process that information and decide what to do with it. In this chapter you will:

* Build a feed-forward network to process context.
* Use a GELU curve to help the model learn smoothly.
* Add Layer Normalization to keep the math stable.

**Words to Know**
    - **Feed-Forward**: A small neural network that processes each token independently.
    - **GELU**: A smooth curve that replaces negative numbers with near-zero values.
    - **LayerNorm**: A step that resets numbers to a safe size so training stays stable.

## Theory

### Thinking on Your Own

Attention is like listening to your team explain their ideas. Feed-forward is going back to your desk and thinking it over yourself. 

![A token goes through a wide workspace to process information](../assets/diagrams/ch08-feedforward-reasoning.png){ width="658" }
*Figure 8.2: The network expands to think, then compresses back to an answer.*

The feed-forward layer is a small neural network applied to each token by itself (Figure 8.2). In our model, a token is a list of 128 numbers. The network first expands this to 512 numbers. This 4x expansion gives the model a wide workspace to spread out its calculations. After processing, it compresses the answer back down to 128 numbers. 

### The Smooth GELU Curve

Inside the feed-forward network, we apply a mathematical curve called GELU. 

![ReLU creates a sharp corner while GELU makes a gentle curve](../assets/diagrams/ch08-gelu-curve.png){ width="418" }
*Figure 8.3: GELU replaces a hard switch with a smooth dimmer.*

A neural network needs a non-linear curve to learn complex patterns. An older choice is ReLU, which acts like a hard switch: negative numbers become exactly zero, while positive numbers stay the same. 

GELU is a gradual dimmer switch, as seen in Figure 8.3. Strongly negative numbers become nearly zero. Numbers near zero are gently reduced. Positive numbers pass through almost unchanged. 

During training, we use calculus to figure out how to adjust the model's weights. A hard corner like ReLU can produce sudden jumps that destabilize the math. GELU's smooth curve makes training much more stable. Most modern models use it.

### Keeping Numbers Safe with LayerNorm

As numbers flow through many layers of a neural network, they can grow very large or shrink very small. Like compound interest, small multiplications add up quickly. Extremely large or small numbers break the training math.

Layer Normalization (LayerNorm) fixes this. It re-centers and re-scales the numbers for each token. After LayerNorm, the values average to 0 and have a standard spread of 1. 

![A vector of varying numbers is scaled to have zero mean and standard variance](../assets/diagrams/ch08-layernorm.png){ width="578" }
*Figure 8.4: LayerNorm centers messy numbers back around zero.*

Think of it like resetting a runner's stopwatch after every lap (Figure 8.4). It does not change who is winning, but it keeps the numbers on the screen small and easy to read. In modern models, we apply LayerNorm *before* each major step. This gives the attention and feed-forward layers well-behaved numbers to work with.

Returning to the big picture map in Figure 8.1, the feed-forward network processes the context gathered by attention, and LayerNorm ensures the math remains stable before we pass the numbers to the next stage.

## Code

```python
class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        C = config.n_embd
        # A small neural network applied to each token independently
        self.net = nn.Sequential(
            nn.Linear(C, 4 * C),
            nn.GELU(),
            nn.Linear(4 * C, C),
            nn.Dropout(config.dropout),
        )

    def forward(self, x):
        return self.net(x)
```

Run the script to see the feed-forward layer and LayerNorm in action.

```python
$ python src/ch07_feedforward.py
--- GELU activation ---
GELU is like ReLU (zeros out negatives) but with a smooth curve.

  Input : [-3.0, -1.0, -0.5, 0.0, 0.5, 1.0, 2.0, 3.0]
  GELU  : [-0.004, -0.159, -0.154, 0.0, 0.346, 0.841, 1.954, 2.996]
  ReLU  : [0.0, 0.0, 0.0, 0.0, 0.5, 1.0, 2.0, 3.0]

--- Layer Normalization ---
LayerNorm re-centers and re-scales values at each position.

Before LayerNorm: mean=5.23, std=9.13
After  LayerNorm: mean=0.0000, std=1.0039
```

**What just happened:**

1. Lines 7 and 9 build a feed-forward network that expands from 128 to 512, then back to 128.
2. Line 8 applies GELU, which gently reduces negative numbers instead of snapping them to zero.
3. We passed messy numbers through LayerNorm and saw them neatly centered at zero with a standard spread of 1.

**Shape Check:**

- Input to FeedForward: `[Batch, Time, 128]`
- Output of FeedForward: `[Batch, Time, 128]`

## Try It

**Try It**
    Open `src/ch07_feedforward.py` and change the LayerNorm input to have a massive spread: `x_single = torch.randn(config.n_embd) * 1000 + 500`. Run the script again. You will see that LayerNorm still perfectly tames it back to a mean of 0 and a standard spread of 1.

**In Business**
    Imagine you are building an assistant to draft company emails in your house style. If the system is unstable, it might output gibberish. LayerNorm acts like a manager double-checking work between steps, ensuring the data never drifts too far off track before passing it to the next team.

## Key Takeaways

* The feed-forward layer processes each token independently to build meaning.
* It expands the token's data 4x to give itself room to think.
* GELU provides a smooth mathematical curve to make training stable.
* LayerNorm resets numbers to a safe size so deep networks do not break.

## Check Your Understanding

1. Why does the feed-forward network expand its input size by 4?
2. What happens to a strongly negative number when it passes through GELU?
3. What is the average value of a token's numbers immediately after LayerNorm?


## Further Reading

**The line that keeps training stable.** As numbers pass through a deep stack they drift, some growing, some shrinking, until learning stalls. Layer normalization rescales each token's vector back to a standard spread before the next step, using only that token's own numbers, so it works the same whatever the batch size. It is one line in Chapter 8 and one of the reasons deep Transformers train at all.

**The smooth switch inside the feed-forward layer.** A network needs a non-linear step, or every layer collapses into one. The common choice cut everything negative to zero, a hard switch. GELU fades instead of cutting, which gives the optimizer a gentler surface to work on and is the activation used in GPT-style models, including the one in Chapter 8.

<div class="refs" markdown>

Ba, J. L., Kiros, J. R., & Hinton, G. E. (2016). *Layer normalization* (arXiv:1607.06450). arXiv. https://doi.org/10.48550/arXiv.1607.06450

Hendrycks, D., & Gimpel, K. (2016). *Gaussian error linear units (GELUs)* (arXiv:1606.08415). arXiv. https://doi.org/10.48550/arXiv.1606.08415

</div>

---

### `src/ch07_feedforward.py`

The whole file, ready to edit and run.

In [ ]:
__file__ = "src/ch07_feedforward.py"   # a cell has none, and the file uses it to find the text

"""
Implement the feed-forward network and layer normalization.
This file belongs to Chapter 8.
Run: python src/ch07_feedforward.py
"""
import torch
import torch.nn as nn
import torch.nn.functional as F
import os
import sys

from src.utils.config import GPTConfig

# Settings
config = GPTConfig()

# --- The Idea ---
class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        C = config.n_embd
        # A small neural network applied to each token independently
        self.net = nn.Sequential(
            nn.Linear(C, 4 * C),
            nn.GELU(),
            nn.Linear(4 * C, C),
            nn.Dropout(config.dropout),
        )

    def forward(self, x):
        return self.net(x)

# --- Demo ---
if __name__ == "__main__":
    torch.manual_seed(42)
    print("Chapter 8: Feed-Forward Layer and Layer Norm\n")

    ff = FeedForward()
    total_params = sum(p.numel() for p in ff.parameters())
    print(f"FeedForward parameters: {total_params:,}")
    print(f"  (C={config.n_embd} -> 4C={4*config.n_embd} -> C={config.n_embd})")

    B, T = 2, 10
    x = torch.randn(B, T, config.n_embd)
    out = ff(x)
    print(f"\nInput  shape: {x.shape}")
    print(f"Output shape: {out.shape}   (same shape as input)")

    print("\n--- GELU activation ---")
    print("GELU is like ReLU (zeros out negatives) but with a smooth curve.")
    sample = torch.tensor([-3.0, -1.0, -0.5, 0.0, 0.5, 1.0, 2.0, 3.0])
    gelu_out = F.gelu(sample)
    relu_out = F.relu(sample)
    print(f"\n  Input : {sample.tolist()}")
    print(f"  GELU  : {[round(v, 3) for v in gelu_out.tolist()]}")
    print(f"  ReLU  : {relu_out.tolist()}")

    print("\n--- Layer Normalization ---")
    print("LayerNorm re-centers and re-scales values at each position.")

    # LayerNorm ensures each token's vector has mean 0 and variance 1
    # initially
    ln = nn.LayerNorm(config.n_embd)
    x_single = torch.randn(config.n_embd) * 10 + 5
    x_normed = ln(x_single)

    print(f"\nBefore LayerNorm: mean={x_single.mean():.2f}, "
          f"std={x_single.std():.2f}")
    print(f"After  LayerNorm: mean={x_normed.mean():.4f}, "
          f"std={x_normed.std():.4f}")
    print("(After normalization: mean ~0, std ~1)")

    print("\nFeed-forward and LayerNorm done! Ready for Chapter 9.")